## 01 — Bronze Layer: Raw Data Ingestion
Reads CSV files from the DBFS volume and writes to Delta tables in the Bronze schema.
Each table is loaded as-is (no type casting) with ingestion timestamp and source file metadata.

In [ ]:
# pyright: reportMissingImports=false, reportUndefinedVariable=false
from pyspark.sql.functions import current_timestamp, col

### Configuration

In [ ]:
catalog = "workspace"
bronze_schema = "indicium_bronze"
base_path = "/Volumes/workspace/indicium_bronze/raw_data"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}")
print(f"Schema ready: {catalog}.{bronze_schema}")

### Source Tables

In [ ]:
source_tables = [
    "categories",
    "customers",
    "customer_customer_demo",
    "customer_demographics",
    "employees",
    "employee_territories",
    "orders",
    "order_details",
    "products",
    "region",
    "shippers",
    "suppliers",
    "territories",
    "us_states",
]

### Ingestion Loop

In [ ]:
for table_name in source_tables:
    file_path = f"{base_path}/{table_name}.csv"

    df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("delimiter", ";")
        .option("inferSchema", "false")
        .load(file_path)
        .withColumn("ingestion_ts", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
    )

    target = f"{catalog}.{bronze_schema}.{table_name}"
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )

    print(f"Loaded: {target} ({df.count():,} rows)")

### Row Count Validation

In [ ]:
for table_name in source_tables:
    target = f"{catalog}.{bronze_schema}.{table_name}"
    row_count = spark.table(target).count()
    print(f"{target}: {row_count:,} rows")